# Force density: compas_fd

In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps
%pip install -q compas_fd==0.5.4
%pip install -q https://brg-teaching.github.io/Teaching/CAS/website/session-viewer/session_py-0.260.0-py3-none-any.whl

In [ ]:
from IPython.display import display
from compas.datastructures import Mesh
from compas_fd.solvers import fd_numpy
from compas_notebook import Viewer
from compas_notebook.config import Config
from session_py import Color, Line, Point, Session, show
from session_py.viewer import supports_webgpu

## Cable net

In [ ]:
mesh = Mesh.from_meshgrid(dx=10, nx=10)
fixed = list(mesh.vertices_where(vertex_degree=2))
edges = list(mesh.edges())
q = [10.0 if mesh.is_edge_on_boundary(edge) else 1.0 for edge in edges]
loads = [[0, 0, -0.2]] * mesh.number_of_vertices()

## Solve

In [ ]:
result = fd_numpy(vertices=mesh.vertices_attributes("xyz"), fixed=fixed, edges=edges, forcedensities=q, loads=loads)
for vertex, xyz in zip(mesh.vertices(), result.vertices):
    mesh.vertex_attributes(vertex, "xyz", xyz)
result.vertices[:, 2].min()

## Session Viewer

In [ ]:
if await supports_webgpu():
    session = Session("cable_net")
    nodes = [Point(*xyz, name=f"n{index}") for index, xyz in enumerate(mesh.vertices_attributes("xyz"))]
    for node in nodes:
        node.width = 4.0
        session.add_point(node, None)
    for index, (start, end) in enumerate(edges):
        line = Line.from_points(nodes[start], nodes[end])
        line.name = f"e{index}"
        line.width = 2.0
        line.linecolor = Color(220 / 255, 75 / 255, 141 / 255)
        session.add_line(line, None)
    display(show(session, height=420))
else:
    config = Config()
    config.view.width, config.view.height = 700, 360
    config.view.camera.position = [5, -10, 6]
    config.view.camera.target = [5, 5, -1]
    viewer = Viewer(config=config)
    viewer.scene.add(mesh, color="#dc4b8d", show_edges=True)
    viewer.show()